# Multilingual Document OCR + MRZ Parser
### Domino Data Lab · Arabic / French / English · Fully Offline (v4)

---

## Bugs fixed in this version

| # | Error message | Root cause | Fix |
|---|--------------|-----------|-----|
| 1 | `AssertionError: param lang must in dict_keys, but got ar,fr,en` | PaddleOCR only accepts ONE language per instance | Two engines: `lang='ar'` + `lang='fr'` |
| 2 | `ConnectionError` downloading `ch_PP-OCRv4_det_infer.tar` even after passing `det_model_dir` | PaddleOCR ignored `det_model_dir` and fell back to downloading its defaults | Models placed inside `~/.paddleocr/whl/` — the EXACT cache path PaddleOCR checks first |

## Before running this notebook

Complete Steps 1–4 in **INSTRUCTIONS.txt** first:
1. **Laptop terminal** — download model `.tar` files  
2. **Domino browser** — upload models + images  
3. **Domino terminal** — `pip install` packages  
4. **Domino terminal** — `cp` models into `~/.paddleocr/whl/`  ← **the critical step**

## Expected cache structure on Domino after Step 4
```
~/.paddleocr/whl/det/multilingual/Multilingual_PP-OCRv3_det_infer/
~/.paddleocr/whl/rec/arabic/arabic_PP-OCRv3_rec_infer/
~/.paddleocr/whl/rec/latin/latin_PP-OCRv3_rec_infer/
```

---
## Cell 1 — Imports

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import json
import logging
import os
import re
import shutil
import time
from pathlib import Path
from typing import Any

# ── Third-party ───────────────────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

print('✓ Cell 1 complete — all imports OK')

---
## Cell 2 — Configuration and Model Verification

This cell:
1. Defines all paths
2. Checks that models are in `~/.paddleocr/whl/` (the cache PaddleOCR checks first)
3. If models are in `/mnt/data/models/` but NOT yet in the cache, copies them automatically

> If you see **red error text**, go back to INSTRUCTIONS.txt Steps 2–4.

In [ ]:
# =============================================================================
# PATHS — edit only if your folder names are different
# =============================================================================

# Where your document images live on Domino
INPUT_DIR    = Path('/mnt/data/documents')

# Where output files will be saved (created automatically)
OUTPUT_JSON  = Path('/mnt/artifacts/results.json')
OUTPUT_EXCEL = Path('/mnt/artifacts/report.xlsx')

# Where you uploaded the extracted model folders in Step 2
UPLOADED_MODELS = Path('/mnt/data/models')

# =============================================================================
# PADDLEOCR CACHE STRUCTURE
# PaddleOCR checks ~/.paddleocr/whl/ BEFORE trying to download anything.
# If models exist here, NO network call is made.
# =============================================================================

PADDLE_CACHE    = Path.home() / '.paddleocr' / 'whl'

# Exact sub-paths PaddleOCR uses internally for these models
CACHE_DET   = PADDLE_CACHE / 'det'  / 'multilingual' / 'Multilingual_PP-OCRv3_det_infer'
CACHE_AR    = PADDLE_CACHE / 'rec'  / 'arabic'       / 'arabic_PP-OCRv3_rec_infer'
CACHE_LATIN = PADDLE_CACHE / 'rec'  / 'latin'        / 'latin_PP-OCRv3_rec_infer'

# Source folders (uploaded in Step 2)
SRC_DET     = UPLOADED_MODELS / 'Multilingual_PP-OCRv3_det_infer'
SRC_AR      = UPLOADED_MODELS / 'arabic_PP-OCRv3_rec_infer'
SRC_LATIN   = UPLOADED_MODELS / 'latin_PP-OCRv3_rec_infer'

# =============================================================================
# AUTO-COPY: if models are not in the cache yet, copy them now
# (same as running the cp commands in Step 4 of INSTRUCTIONS.txt)
# =============================================================================

def ensure_in_cache(src: Path, dst: Path, label: str) -> None:
    """
    Copy model folder from uploaded location to PaddleOCR cache if not there.
    If already in cache, do nothing.
    """
    if dst.exists() and any(dst.iterdir()):
        print(f'  ✓ {label:30s} already in cache')
        return
    if not src.exists():
        raise FileNotFoundError(
            f'Source model not found: {src}\n'
            f'Complete Steps 2 (upload) in INSTRUCTIONS.txt.'
        )
    print(f'  → Copying {label} to cache …')
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(src), str(dst))
    print(f'  ✓ {label:30s} copied to cache')


print('Checking PaddleOCR model cache …')
ensure_in_cache(SRC_DET,   CACHE_DET,   'Detection model')
ensure_in_cache(SRC_AR,    CACHE_AR,    'Arabic rec model')
ensure_in_cache(SRC_LATIN, CACHE_LATIN, 'Latin rec model  (fr/en)')

# =============================================================================
# PERFORMANCE
# =============================================================================

MAX_DIMENSION = 1280
CPU_THREADS   = os.cpu_count() or 4

# =============================================================================
# CONSTANTS
# =============================================================================

SUPPORTED_EXT    = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
MRZ_LINE_PATTERN = re.compile(r'^[A-Z0-9<]{30,}$')

print()
print('✓ Cell 2 complete — configuration OK')
print(f'  Cache root    : {PADDLE_CACHE}')
print(f'  Input images  : {INPUT_DIR}')
print(f'  CPU threads   : {CPU_THREADS}')

---
## Cell 3 — Image Preprocessing

In [ ]:
def load_and_resize(
    image_path: str | Path,
    max_dim: int = MAX_DIMENSION,
) -> np.ndarray:
    """
    Load an image and downscale it if the longest side exceeds max_dim.

    Why resize?
        Halves CPU inference time on large scans (e.g. 3000px phone photos)
        while keeping text fully readable at 1280px.
        cv2.INTER_AREA gives best quality when shrinking.

    Parameters
    ----------
    image_path : path to the image
    max_dim    : maximum side in pixels (default 1280)

    Returns
    -------
    BGR numpy array ready for PaddleOCR
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f'Cannot read image: {image_path}')

    h, w  = img.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)   # never upscale

    if scale < 1.0:
        new_w = int(w * scale)
        new_h = int(h * scale)
        img   = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        log.debug('Resized %s  %dx%d → %dx%d',
                  Path(image_path).name, w, h, new_w, new_h)

    return img


print('✓ Cell 3 complete — load_and_resize defined')

---
## Cell 4 — Build the Two OCR Engines

### Why two engines?
PaddleOCR only accepts **one language code per instance**.  
Passing `lang='ar,fr,en'` raises `AssertionError`.  
We create one engine per script family and merge their outputs:

| Engine | `lang` | Recognises |
|--------|--------|------------|
| `ocr_ar` | `'ar'` | Arabic script |
| `ocr_latin` | `'fr'` | French AND English (Latin script) |

### Why no more `det_model_dir` / `rec_model_dir`?
Those parameters were **being ignored** by PaddleOCR, which then tried to
download its own defaults from the internet.  
The fix (done in Cell 2) is to place models inside `~/.paddleocr/whl/` —
PaddleOCR checks that folder **before** attempting any download.  
We just call `PaddleOCR(lang='ar', ...)` and it finds the files automatically.

> ⏱ Takes 10–30 seconds. No download. No network call.

In [ ]:
from paddleocr import PaddleOCR


def _make_engine(lang: str) -> PaddleOCR:
    """
    Create a single-language PaddleOCR engine.

    Models are read from ~/.paddleocr/whl/ (populated in Cell 2).
    PaddleOCR checks the cache FIRST — no network call is made.

    Parameters
    ----------
    lang : str
        Single PaddleOCR language code.
        'ar' → Arabic model
        'fr' → Latin model (French + English)

    Returns
    -------
    PaddleOCR instance (offline)
    """
    return PaddleOCR(
        lang=lang,
        use_angle_cls=False,    # documents assumed upright — saves time
        use_gpu=False,          # CPU-only
        enable_hpi=True,        # ONNX Runtime High-Performance Inference
        cpu_threads=CPU_THREADS,
        show_log=False,
    )


# ── Arabic engine (reads from ~/.paddleocr/whl/rec/arabic/) ──────────────────
print('Initialising Arabic engine  (lang="ar") …')
print('Reading from cache — no download …')
ocr_ar = _make_engine('ar')
print('  ✓ Arabic engine ready')

# ── Latin engine  (reads from ~/.paddleocr/whl/rec/latin/) ───────────────────
print('Initialising Latin engine   (lang="fr") …')
print('Reading from cache — no download …')
ocr_latin = _make_engine('fr')
print('  ✓ Latin engine ready  (French + English)')

print()
print('✓ Cell 4 complete — both OCR engines ready, fully offline')

---
## Cell 5 — MRZ Detection and Parsing

In [ ]:
def detect_mrz_lines(text_lines: list[str]) -> list[str]:
    """
    Scan OCR text lines for MRZ-like content using a regex heuristic.

    ICAO MRZ lines contain only:
      - uppercase letters A-Z
      - digits 0-9
      - '<' filler character
    Minimum length: 30 characters (TD1=30, TD2=36, passport TD3=44).

    OCR sometimes inserts spaces inside MRZ lines — removed before matching.

    Parameters
    ----------
    text_lines : list of raw OCR text lines

    Returns
    -------
    list of cleaned MRZ line strings
    """
    mrz_lines: list[str] = []
    for line in text_lines:
        cleaned = line.strip().replace(' ', '')
        if MRZ_LINE_PATTERN.match(cleaned):
            mrz_lines.append(cleaned)
    return mrz_lines


def parse_mrz_with_omnimrz(mrz_lines: list[str]) -> dict[str, Any]:
    """
    Parse and validate MRZ lines using OmniMRZ (ICAO 9303 standard).

    Parameters
    ----------
    mrz_lines : MRZ lines from detect_mrz_lines()

    Returns
    -------
    dict with keys:
        raw_mrz        — original MRZ line list
        parsed_fields  — ICAO 9303 structured fields
        validation     — per-field checksum results + overall_valid
        error          — only present if parsing failed
    """
    try:
        from omnimrz import MRZ
    except ImportError:
        log.warning('omnimrz not installed — MRZ structural parsing skipped.')
        return {'error': 'omnimrz not installed', 'raw_mrz': mrz_lines}

    mrz_string = '\n'.join(mrz_lines)
    try:
        mrz_obj = MRZ(mrz_string)

        fields: dict[str, Any] = {
            'document_type':    getattr(mrz_obj, 'document_type',    None),
            'issuing_country':  getattr(mrz_obj, 'issuing_country',  None),
            'document_number':  getattr(mrz_obj, 'document_number',  None),
            'surname':          getattr(mrz_obj, 'surname',          None),
            'given_names':      getattr(mrz_obj, 'given_names',      None),
            'nationality':      getattr(mrz_obj, 'nationality',      None),
            'birth_date':       getattr(mrz_obj, 'birth_date',       None),
            'sex':              getattr(mrz_obj, 'sex',              None),
            'expiry_date':      getattr(mrz_obj, 'expiry_date',      None),
            'optional_data_1':  getattr(mrz_obj, 'optional_data_1', None),
            'optional_data_2':  getattr(mrz_obj, 'optional_data_2', None),
        }

        validation: dict[str, Any] = {}
        for attr in dir(mrz_obj):
            if 'valid' in attr.lower() or 'check' in attr.lower():
                try:
                    validation[attr] = getattr(mrz_obj, attr)
                except Exception:
                    pass

        overall = getattr(mrz_obj, 'valid', None)
        if overall is None:
            overall = all(v for v in validation.values() if isinstance(v, bool))
        validation['overall_valid'] = overall

        return {
            'raw_mrz':       mrz_lines,
            'parsed_fields': fields,
            'validation':    validation,
        }

    except Exception as exc:
        log.warning('OmniMRZ parsing failed: %s', exc)
        return {'raw_mrz': mrz_lines, 'error': str(exc)}


print('✓ Cell 5 complete — MRZ helpers defined')

---
## Cell 6 — Document Processor (dual engine)

In [ ]:
def _extract_lines(engine: Any, img: np.ndarray) -> list[str]:
    """
    Run one PaddleOCR engine on img and return a flat list of text strings.

    PaddleOCR raw output structure:
        [ [ [bounding_box, (text, confidence)], ... ] ]
    This unpacks it into a plain list of strings.
    """
    lines: list[str] = []
    raw = engine.ocr(img, cls=False)
    if raw and raw[0]:
        for item in raw[0]:
            if item and len(item) >= 2:
                txt = (
                    item[1][0]
                    if isinstance(item[1], (list, tuple))
                    else str(item[1])
                )
                lines.append(txt)
    return lines


def process_document(
    image_path:   str | Path,
    engine_ar:    Any,
    engine_latin: Any,
    max_dim:      int = MAX_DIMENSION,
) -> dict[str, Any]:
    """
    Run OCR on one document image using both engines and optionally parse MRZ.

    Steps:
      1. Load and resize the image.
      2. engine_ar   runs on image → Arabic text lines
      3. engine_latin runs on image → French/English text lines
      4. Merge both lists, removing exact duplicates. Arabic lines come first.
      5. Scan merged lines for MRZ pattern (ICAO heuristic).
      6. If >= 2 MRZ lines found: parse with OmniMRZ.

    Parameters
    ----------
    image_path   : path to the document image
    engine_ar    : Arabic PaddleOCR engine   (lang='ar')
    engine_latin : Latin  PaddleOCR engine   (lang='fr')
    max_dim      : resize threshold in pixels

    Returns
    -------
    dict with keys:
        file_name   — filename
        full_text   — all OCR text, newline-separated
        has_mrz     — True if MRZ was detected
        mrz_data    — parsed MRZ dict or None
        error       — error message or None
        elapsed_sec — processing time in seconds
    """
    image_path = Path(image_path)
    result: dict[str, Any] = {
        'file_name':   image_path.name,
        'full_text':   '',
        'has_mrz':     False,
        'mrz_data':    None,
        'error':       None,
        'elapsed_sec': 0.0,
    }
    t0 = time.perf_counter()

    try:
        # 1. Load image
        img = load_and_resize(image_path, max_dim)

        # 2+3. Run both engines
        lines_ar    = _extract_lines(engine_ar,    img)
        lines_latin = _extract_lines(engine_latin, img)

        # 4. Merge, Arabic first, remove exact duplicates
        seen: set[str]        = set()
        text_lines: list[str] = []
        for line in lines_ar + lines_latin:
            if line not in seen:
                seen.add(line)
                text_lines.append(line)

        result['full_text'] = '\n'.join(text_lines)
        log.debug('%s  ar=%d  latin=%d  merged=%d',
                  image_path.name, len(lines_ar), len(lines_latin), len(text_lines))

        # 5+6. MRZ detection and parsing
        mrz_lines = detect_mrz_lines(text_lines)
        if len(mrz_lines) >= 2:
            result['has_mrz']  = True
            result['mrz_data'] = parse_mrz_with_omnimrz(mrz_lines)
            log.info('  ✓ MRZ detected (%d lines)', len(mrz_lines))
        else:
            log.info('  — No MRZ found')

    except Exception as exc:
        log.error('Error processing %s: %s', image_path.name, exc)
        result['error'] = str(exc)

    result['elapsed_sec'] = round(time.perf_counter() - t0, 3)
    return result


print('✓ Cell 6 complete — process_document defined (dual engine)')

---
## Cell 7 — Test with One Sample Image
Edit `SAMPLE_IMAGE` to point to one of your documents.  
Good sanity check before running the full batch.

In [ ]:
# ── Edit this to one of your image files ─────────────────────────────────────
SAMPLE_IMAGE = Path('/mnt/data/documents/sample.jpg')   # ← change me

if SAMPLE_IMAGE.exists():
    print(f'Testing with: {SAMPLE_IMAGE.name}')
    test_result = process_document(SAMPLE_IMAGE, ocr_ar, ocr_latin)

    print(f'\n--- RESULT ---')
    print(f'File       : {test_result["file_name"]}')
    print(f'Has MRZ    : {test_result["has_mrz"]}')
    print(f'Time       : {test_result["elapsed_sec"]}s')
    print(f'Error      : {test_result["error"]}')
    print(f'\n--- FULL OCR TEXT ---')
    print(test_result['full_text'] or '(no text detected)')

    if test_result['has_mrz']:
        print(f'\n--- MRZ DATA ---')
        print(json.dumps(test_result['mrz_data'],
                         ensure_ascii=False, indent=2, default=str))

    print('\n✓ Cell 7 complete')
else:
    print(f'[SKIP] {SAMPLE_IMAGE} does not exist.')
    print('Edit SAMPLE_IMAGE path above, or go directly to Cell 8.')

---
## Cell 8 — Batch Processing
Processes every supported image under `INPUT_DIR` (set in Cell 2).  
One progress bar tick per document.

In [ ]:
def collect_images(input_path: Path) -> list[Path]:
    """
    Return a sorted list of image paths.
    Accepts a single file OR a directory (searched recursively).
    """
    if input_path.is_file():
        return [input_path]
    images = sorted(
        p for p in input_path.rglob('*')
        if p.suffix.lower() in SUPPORTED_EXT
    )
    if not images:
        print(f'WARNING: no images found in {input_path}')
    return images


images = collect_images(INPUT_DIR)
print(f'Found {len(images)} image(s) in {INPUT_DIR}')
for p in images:
    print(f'  {p.name}')

print()
all_results: list[dict[str, Any]] = []

for img_path in tqdm(images, desc='OCR + MRZ', unit='doc'):
    log.info('Processing: %s', img_path.name)
    r = process_document(img_path, ocr_ar, ocr_latin)
    all_results.append(r)
    log.info('  Done %.2fs | has_mrz=%s | error=%s',
             r['elapsed_sec'], r['has_mrz'], r['error'])

total_time = sum(r['elapsed_sec'] for r in all_results)
print(f'\n✓ Cell 8 complete — {len(all_results)} document(s) in {total_time:.1f}s')

---
## Cell 9 — Save Results as JSON

In [ ]:
def save_json(results: list[dict[str, Any]], output_path: Path) -> None:
    """Write results to a pretty-printed UTF-8 JSON file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2, default=str)
    print(f'  Saved {output_path.stat().st_size/1024:.1f} KB')


save_json(all_results, OUTPUT_JSON)
print(f'✓ Cell 9 complete — JSON saved → {OUTPUT_JSON}')

---
## Cell 10 — Save Results as Excel

| Sheet | Content |
|-------|---------|
| **Summary** | One row per document — filename, has_mrz, valid, time, error |
| **MRZ Details** | All ICAO 9303 parsed fields for docs with MRZ |
| **Full Text** | Complete OCR text for every document |

In [ ]:
def _apply_header_style(ws) -> None:
    fill = PatternFill(fill_type='solid', fgColor='BDD7EE')
    for cell in ws[1]:
        cell.font      = Font(bold=True)
        cell.fill      = fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)


def _autofit_columns(ws, max_width: int = 60) -> None:
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        best   = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[letter].width = min(best + 4, max_width)


def build_excel(results: list[dict[str, Any]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    summary_rows: list[dict] = []
    mrz_rows:     list[dict] = []
    text_rows:    list[dict] = []

    for r in results:
        mrz_data   = r.get('mrz_data') or {}
        validation = mrz_data.get('validation', {}) if r.get('has_mrz') else {}
        overall    = validation.get('overall_valid')

        summary_rows.append({
            'file_name':         r.get('file_name'),
            'has_mrz':           r.get('has_mrz'),
            'overall_mrz_valid': overall,
            'elapsed_sec':       r.get('elapsed_sec'),
            'error':             r.get('error'),
        })

        text_rows.append({
            'file_name': r.get('file_name'),
            'full_text': r.get('full_text'),
        })

        if r.get('has_mrz') and mrz_data:
            pf = mrz_data.get('parsed_fields', {}) or {}
            mrz_rows.append({
                'file_name':       r.get('file_name'),
                'raw_mrz':         ' | '.join(mrz_data.get('raw_mrz', [])),
                'document_type':   pf.get('document_type'),
                'issuing_country': pf.get('issuing_country'),
                'document_number': pf.get('document_number'),
                'surname':         pf.get('surname'),
                'given_names':     pf.get('given_names'),
                'nationality':     pf.get('nationality'),
                'birth_date':      pf.get('birth_date'),
                'sex':             pf.get('sex'),
                'expiry_date':     pf.get('expiry_date'),
                'optional_data_1': pf.get('optional_data_1'),
                'optional_data_2': pf.get('optional_data_2'),
                'overall_valid':   validation.get('overall_valid'),
                **{
                    f'valid_{k}': v
                    for k, v in validation.items()
                    if k != 'overall_valid'
                },
            })

    df_summary = pd.DataFrame(summary_rows)
    df_mrz     = pd.DataFrame(mrz_rows) if mrz_rows else pd.DataFrame(columns=['file_name'])
    df_text    = pd.DataFrame(text_rows)

    with pd.ExcelWriter(str(output_path), engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Summary',     index=False)
        df_mrz.to_excel(    writer, sheet_name='MRZ Details', index=False)
        df_text.to_excel(   writer, sheet_name='Full Text',   index=False)

    wb = load_workbook(str(output_path))
    for ws in wb.worksheets:
        _apply_header_style(ws)
        _autofit_columns(ws)
    wb.save(str(output_path))
    print(f'  Saved {output_path.stat().st_size/1024:.1f} KB')


build_excel(all_results, OUTPUT_EXCEL)
print(f'✓ Cell 10 complete — Excel saved → {OUTPUT_EXCEL}')

---
## Cell 11 — Final Summary

In [ ]:
df_s        = pd.read_excel(str(OUTPUT_EXCEL), sheet_name='Summary')
total_time  = sum(r['elapsed_sec'] for r in all_results)
n           = len(df_s)

print('=' * 52)
print('  FINAL RESULTS SUMMARY')
print('=' * 52)
print(f'  Total documents processed : {n}')
print(f'  Documents with MRZ        : {int(df_s["has_mrz"].sum())}')
print(f'  MRZ passed validation     : {int(df_s["overall_mrz_valid"].sum())}')
print(f'  Processing errors         : {int(df_s["error"].notna().sum())}')
print(f'  Total processing time     : {total_time:.1f}s')
print(f'  Average per document      : {total_time/max(n,1):.2f}s')
print('=' * 52)
print(f'  JSON  → {OUTPUT_JSON}')
print(f'  Excel → {OUTPUT_EXCEL}')
print('=' * 52)
print()
print('Download files from: Domino browser → Jobs → Artifacts')
print()
df_s